# Modul B · Kapitel 2 · Bonus — Navigable Small World und HNSW

## Challenge: Den Index einer Vector Database selbst bauen


**Lernziel:** Du kannst erklären und selbst nachbauen, wie eine Vector Database Nachbarn findet,
ohne alle Abstände zu rechnen — und du kannst benennen, was man dafür aufgibt.

Dieses Notebook ist freiwillig. Es geht eine Ebene tiefer als der Rest des Kapitels: nicht
*wie benutze ich* eine Vector Database, sondern *was passiert darin*.

```
Vektoren ──► Graph ──► Greedy-Suche ──► Kandidatenliste ──► Ebenen ──► HNSW
```

Gerechnet wird auf den 90 Chunks der Wissensbasis und auf einem synthetischen Datensatz aus
8.000 Vektoren. Ein Sprachmodell wird nicht gebraucht — alles hier ist Geometrie und
Buchhaltung.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **4 Challenges**.


---
## 0 · Setup

▶️ Führe die beiden nächsten Zellen aus.

Dieses Notebook rechnet nur mit `numpy` und zeichnet mit `matplotlib`. Ein Sprachmodell wird
nicht angefragt: Die Embeddings der 90 Chunks liegen fertig in `daten/embedding_cache.json`,
`helfer.embed()` holt sie von dort. In Google Colab läuft das Notebook deshalb ohne lokales
Ollama.

`chromadb` kommt erst im letzten Abschnitt dazu, für den Vergleich mit einer echten Vector
Database.


In [ ]:
# ▶️ Pakete
import heapq
import sys
import time
from collections import deque
from pathlib import Path

try:
    import chromadb
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    %pip install -q chromadb matplotlib numpy
    import chromadb
    import matplotlib.pyplot as plt
    import numpy as np

import matplotlib.ticker as ticker

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print(f"numpy {np.__version__}, chromadb {chromadb.__version__}")
print("Setup fertig ✔")


▶️ Die zweite Zelle lädt die 90 Chunks und ihre Embeddings.

Alle Vektoren werden dabei auf Länge 1 gebracht. Das ist keine Kosmetik: Für normalisierte
Vektoren ist die Kosinus-Distanz genau `1 - Skalarprodukt`. Eine Distanz kostet damit ein
`np.dot` — und die Zahl dieser Aufrufe ist die Währung, in der dieses Notebook rechnet.


In [ ]:
# ▶️ helfer.py finden, Chunk-Embeddings laden und normalisieren
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer

chunks = helfer.lade_chunks()
roh = np.asarray(helfer.embed([c["text"] for c in chunks]), dtype=np.float32)
VEKTOREN = roh / np.linalg.norm(roh, axis=1, keepdims=True)

print(f"{len(chunks)} Chunks, {VEKTOREN.shape[1]} Dimensionen")
print(f"Länge des ersten Vektors: {float(np.linalg.norm(VEKTOREN[0])):.4f}")
print(f"Erster Chunk: {chunks[0]['chunk_id']}")


---
## 1 · Die lineare Suche als Bezugspunkt

📖 Ohne Index geht eine Anfrage jeden Vektor der Sammlung durch. Bei 90 Chunks sind das 90
Distanzberechnungen, bei 1.000.000 Chunks eine Million — je Anfrage.

Zwei Zahlen beschreiben jedes Suchverfahren in diesem Notebook:

| Kennzahl | Bedeutung |
|---|---|
| **Distanzberechnungen je Anfrage** | der Rechenaufwand, unabhängig von Sprache und Hardware |
| **Recall** | wie oft das Verfahren dieselben Nachbarn findet wie die lineare Suche |

Die lineare Suche hat Recall 1,0 — sie ist die Wahrheit, gegen die alles andere gemessen wird.
Ihr Aufwand ist die Sammlungsgröße.


In [ ]:
# ▶️ Das Abstandsmaß und die lineare Suche
def distanz(a, b):
    """Kosinus-Distanz zweier normalisierter Vektoren. 0 = gleiche Richtung, 2 = entgegengesetzt."""
    return 1.0 - float(np.dot(a, b))


def suche_linear(vektoren, ziel, k=5):
    """Vergleicht das Ziel mit jedem Vektor.

    Rückgabe: die Indizes der k nächsten Punkte und die Zahl der Distanzberechnungen.
    """
    abstaende = [distanz(v, ziel) for v in vektoren]
    beste = sorted(range(len(vektoren)), key=lambda i: abstaende[i])[:k]
    return beste, len(vektoren)


ziel = VEKTOREN[7]
treffer, berechnungen = suche_linear(VEKTOREN, ziel, k=3)

print(f"Anfrage: {chunks[7]['chunk_id']}")
for rang, i in enumerate(treffer, start=1):
    print(f"  {rang}. {chunks[i]['chunk_id']:<32} Distanz {distanz(VEKTOREN[i], ziel):.4f}")
print(f"Distanzberechnungen: {berechnungen}")


In [ ]:
# ▶️ Laufzeit messen und auf große Sammlungen hochrechnen
t0 = time.perf_counter()
for _ in range(20):
    suche_linear(VEKTOREN, ziel, k=5)
dauer = (time.perf_counter() - t0) / 20
je_vektor = dauer / len(VEKTOREN)
dimension = VEKTOREN.shape[1]

print(f"eine Anfrage über {len(VEKTOREN)} Vektoren: {dauer * 1000:.2f} ms"
      f"   →   je Vektor {je_vektor * 1e6:.1f} µs")
print()
print(f"{'Sammlung':>12}{'Distanzberechnungen':>22}{'Multiplikationen':>20}{'Dauer je Anfrage':>20}")
print("-" * 74)
for groesse in [90, 1_000, 100_000, 1_000_000]:
    print(f"{groesse:>12,}{groesse:>22,}{groesse * dimension:>20,}"
          f"{groesse * je_vektor:>17.3f} s".replace(",", "."))


📖 Das ist das Skalierungsproblem: Bei 1.000.000 Chunks kostet **eine** Anfrage eine Million
Distanzberechnungen, also 768 Millionen Multiplikationen.

Die Sekunden in der letzten Spalte hängen an der Implementierung. Hier steckt in jeder Distanz
ein `np.dot` über 768 float32-Werte. Eine Schleife über dieselben Zahlen in reinem Python ist
mehr als hundertmal langsamer, eine einzige Matrixmultiplikation über die ganze Sammlung
deutlich schneller. An der Steigung ändert keine dieser Varianten etwas: doppelt so viele
Chunks, doppelt so viel Arbeit.

Deshalb zählt dieses Notebook Distanzberechnungen und nicht Millisekunden. Ein Index ändert die
Steigung — der Preis dafür steht weiter unten.


---
## 2 · Der Graph

📖 Die Idee von **Navigable Small World**: Die Vektoren werden vor der ersten Anfrage zu einem
Graphen verbunden. Jeder Punkt ist ein Knoten, jede Kante eine Verbindung zu einem anderen
Punkt. Eine Suche läuft dann von Knoten zu Knoten, statt alle Punkte anzufassen.

Zwei Sorten Kanten gehören dazu:

* **Nahverbindungen.** Jeder Punkt wird mit seinen `m` nächsten Nachbarn verbunden. Das allein
  ergibt einen Nachbarschaftsgraphen: gut zum Feinsuchen, schlecht zum Reisen. Wer am falschen
  Ende startet, hangelt sich Schritt für Schritt durch die halbe Sammlung.
* **Fernverbindungen.** Dazu kommen einige zufällige Kanten zu beliebigen Punkten. Sie
  überspringen große Abstände in einem Schritt.

Erst diese Mischung macht aus dem Nachbarschaftsgraphen ein **Small World**-Netz: Zwischen zwei
beliebigen Punkten liegen wenige Kanten, obwohl fast alle Kanten lokal sind.

Der Graph wird als Dict abgelegt — Schlüssel ist der Index eines Punktes, Wert die sortierte
Liste seiner Nachbarindizes. Ein Dict statt einer Liste, weil die Ebenen weiter unten nur eine
Teilmenge der Punkte enthalten.

▶️ Zwei Werkzeuge, um einen Graphen zu prüfen.


In [ ]:
# ▶️ Erreichbarkeit und Weglängen, beides über Breitensuche
def erreichbar(graph, start=None):
    """Zahl der Knoten, die von `start` aus über Kanten erreichbar sind."""
    start = next(iter(graph)) if start is None else start
    gesehen = {start}
    stapel = [start]
    while stapel:
        knoten = stapel.pop()
        for nachbar in graph[knoten]:
            if nachbar not in gesehen:
                gesehen.add(nachbar)
                stapel.append(nachbar)
    return len(gesehen)


def weglaengen(graph):
    """Mittlere und größte Zahl von Kanten zwischen zwei Knoten des Graphen."""
    summe = paare = weiteste = 0
    for start in graph:
        tiefe = {start: 0}
        schlange = deque([start])
        while schlange:
            knoten = schlange.popleft()
            for nachbar in graph[knoten]:
                if nachbar not in tiefe:
                    tiefe[nachbar] = tiefe[knoten] + 1
                    schlange.append(nachbar)
        summe += sum(tiefe.values())
        paare += len(tiefe) - 1
        weiteste = max(weiteste, max(tiefe.values()))
    return summe / paare, weiteste


print("erreichbar() und weglaengen() stehen bereit")


### 🛠️ Challenge 1: Den Graphen bauen

Schreibe `baue_nsw(vektoren, m=6, fern=2, seed=0)`. Rückgabe ist ein **Dict**
`{index: [nachbarindizes]}`, die Nachbarlisten aufsteigend sortiert und ohne Dubletten.

1. Alle Ähnlichkeiten auf einmal: `aehnlichkeit = vektoren @ vektoren.T`. Anschließend
   `np.fill_diagonal(aehnlichkeit, -2.0)`, damit ein Punkt nicht sein eigener Nachbar wird.
2. Für jeden Punkt `i` die `m` ähnlichsten Punkte bestimmen: `np.argsort(-aehnlichkeit[i])[:m]`.
3. Jede Kante in **beide** Richtungen eintragen — der Graph ist ungerichtet.
4. Danach für jeden Punkt `fern` zufällige Fernverbindungen ergänzen. Der Zufall kommt aus
   `rng = np.random.default_rng(seed)`, die Ziele aus `rng.integers(0, n, size=fern)`. Eine
   Kante auf sich selbst wird übersprungen, sonst gilt auch hier: beide Richtungen.

*Tipp: Sammle die Nachbarn in `kanten = [set() for _ in range(n)]` — Mengen halten die Kanten
von selbst dublettenfrei. Am Ende `{i: sorted(nachbarn) for i, nachbarn in enumerate(kanten)}`.
`np.argsort` liefert `np.int64`, deshalb `int(j)` beim Eintragen.*


In [ ]:
def baue_nsw(vektoren, m=6, fern=2, seed=0):
    """Verbindet jeden Punkt mit seinen m nächsten Nachbarn und mit `fern` zufälligen Punkten."""
    n = len(vektoren)
    rng = np.random.default_rng(seed)
    kanten = [set() for _ in range(n)]

    # Nahverbindungen: die m ähnlichsten Punkte

    # Fernverbindungen: zufällige Ziele, unabhängig vom Abstand

    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 1: baue_nsw() implementieren")


In [ ]:
# ✅ Selbsttest
GRAPH = baue_nsw(VEKTOREN, m=6, fern=2, seed=0)

assert set(GRAPH) == set(range(len(VEKTOREN))), "Jeder Punkt braucht einen Eintrag"
assert all(i not in GRAPH[i] for i in GRAPH), "Kein Punkt ist sein eigener Nachbar"
assert all(len(GRAPH[i]) >= 6 for i in GRAPH), "Jeder Punkt hat mindestens m Nachbarn"
assert all(i in GRAPH[j] for i in GRAPH for j in GRAPH[i]), "Jede Kante gehört in beide Richtungen"
assert all(GRAPH[i] == sorted(set(GRAPH[i])) for i in GRAPH), "Nachbarn sortiert und ohne Dubletten"
assert erreichbar(GRAPH) == len(VEKTOREN), "Der Graph muss zusammenhängend sein"

ohne_fern = baue_nsw(VEKTOREN, m=6, fern=0, seed=0)
assert sum(len(v) for v in ohne_fern.values()) < sum(len(v) for v in GRAPH.values()), \
    "Ohne Fernverbindungen muss der Graph weniger Kanten haben"
assert baue_nsw(VEKTOREN, m=6, fern=2, seed=0) == GRAPH, "Gleicher seed, gleicher Graph"

grade = [len(GRAPH[i]) for i in GRAPH]
print("✅ Challenge 1 gelöst")
print(f"{len(GRAPH)} Knoten, {sum(grade) // 2} Kanten")
print(f"Nachbarn je Punkt: kleinster {min(grade)}, Median {int(np.median(grade))}, größter {max(grade)}")
print(f"Nachbarn von Punkt 0 ({chunks[0]['chunk_id']}): {GRAPH[0]}")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_nsw(vektoren, m=6, fern=2, seed=0):
    """Verbindet jeden Punkt mit seinen m nächsten Nachbarn und mit `fern` zufälligen Punkten."""
    n = len(vektoren)
    rng = np.random.default_rng(seed)
    kanten = [set() for _ in range(n)]

    # Nahverbindungen: die m ähnlichsten Punkte
    aehnlichkeit = vektoren @ vektoren.T
    np.fill_diagonal(aehnlichkeit, -2.0)
    for i in range(n):
        for j in np.argsort(-aehnlichkeit[i])[:m]:
            kanten[i].add(int(j))
            kanten[int(j)].add(i)

    # Fernverbindungen: zufällige Ziele, unabhängig vom Abstand
    for i in range(n):
        for j in rng.integers(0, n, size=fern):
            if int(j) != i:
                kanten[i].add(int(j))
                kanten[int(j)].add(i)

    return {i: sorted(nachbarn) for i, nachbarn in enumerate(kanten)}
```

Der Grad eines Punktes ist größer als `m`: Zu seinen eigenen `m` Kanten kommen die Kanten
derer, die ihn als Nachbarn gewählt haben. Genau deshalb prüft der Selbsttest `>= 6` und nicht
`== 6`.

Der volle `vektoren @ vektoren.T` kostet quadratischen Speicher. Bibliotheken bauen den Graphen
deshalb inkrementell: Jeder neue Punkt sucht sich seine Nachbarn über den bereits bestehenden
Graphen, statt gegen alle zu vergleichen.

</details>


In [ ]:
# ▶️ Was die Fernverbindungen mit den Weglängen machen
print(f"{'m':>3}{'fern':>6}{'Nachbarn (Median)':>20}{'mittlere Weglänge':>20}{'weiteste':>10}")
print("-" * 59)
for m in [3, 6]:
    for fern in [0, 2]:
        g = baue_nsw(VEKTOREN, m=m, fern=fern, seed=0)
        mittel, weiteste = weglaengen(g)
        median = int(np.median([len(g[i]) for i in g]))
        print(f"{m:>3}{fern:>6}{median:>20}{mittel:>20.2f}{weiteste:>10}")


📖 Bei `m=3` ohne Fernverbindungen liegen zwischen zwei Punkten im Mittel 3,8 Kanten, im
schlechtesten Fall 8. Zwei zufällige Kanten je Punkt drücken das auf 2,4 und 4 — bei nur zwei
zusätzlichen Kanten je Punkt. Das ist der Small-World-Effekt.

Bei `m=6` ist der Graph schon dicht genug, dass der Unterschied kleiner ausfällt. Er wächst
wieder, sobald die Sammlung wächst: 90 Punkte liegen alle nah beieinander, 8.000 nicht mehr.


---
## 3 · Die Suche im Graphen

📖 Der Graph steht, jetzt die Suche. Sie ist so einfach, wie sie klingt:

1. An einem Einstiegspunkt beginnen und dessen Distanz zum Ziel berechnen.
2. Alle Nachbarn des aktuellen Punktes durchgehen.
3. Ist einer näher am Ziel, dorthin weitergehen und wieder bei 2 anfangen.
4. Ist keiner näher, anhalten. Der aktuelle Punkt ist das Ergebnis.

Das heißt **Greedy-Suche**: Es wird immer der beste Schritt gemacht, der gerade sichtbar ist,
ohne Rückschau. Angefasst werden nur die Punkte auf dem Weg und deren Nachbarn — nicht die
ganze Sammlung.

▶️ Zum Prüfen zwei kleine Graphen von Hand. Ihre Punkte liegen auf dem Einheitskreis, angegeben
als Winkel. Für zwei Winkel ist die Kosinus-Distanz `1 - cos(Winkeldifferenz)`, also monoton im
Winkelabstand — damit lässt sich jedes Ergebnis im Kopf nachrechnen.


In [ ]:
# ▶️ Zwei Graphen mit bekanntem Ergebnis
def kreis(*winkel):
    """Einheitsvektoren in der Ebene, angegeben als Winkel in Grad."""
    bogen = np.radians(np.array(winkel, dtype=np.float32))
    return np.stack([np.cos(bogen), np.sin(bogen)], axis=1)


# Eine Kette: jeder Punkt kennt nur seinen linken und seinen rechten Nachbarn.
KETTE_PUNKTE = kreis(0, 20, 40, 60, 80, 100)
KETTE = {0: [1], 1: [0, 2], 2: [1, 3], 3: [2, 4], 4: [3, 5], 5: [4]}

# Eine Falle: der beste Punkt (3) hängt hinter einem deutlich schlechteren (2).
FALLE_PUNKTE = kreis(0, 40, 120, 55)
FALLE = {0: [1], 1: [0, 2], 2: [1, 3], 3: [2]}
ZIEL_FALLE = kreis(50)[0]

print("Falle — Distanzen zum Ziel bei 50°:")
for i, winkel in enumerate([0, 40, 120, 55]):
    print(f"  Punkt {i} ({winkel:>3}°): {distanz(FALLE_PUNKTE[i], ZIEL_FALLE):.4f}"
          f"   Nachbarn {FALLE[i]}")


### 🛠️ Challenge 2: Greedy-Suche

Schreibe `suche_greedy(graph, vektoren, ziel, einstieg)`. Rückgabe ist ein Tupel aus drei
Dingen:

| Rückgabe | Inhalt |
|---|---|
| `gefunden` | der Index des Punktes, an dem die Suche stehen bleibt |
| `pfad` | die Liste der besuchten Punkte, vom Einstieg bis zum Ergebnis |
| `berechnungen` | wie oft `distanz()` aufgerufen wurde |

Der Ablauf: Distanz des Einstiegs berechnen (das ist die erste Berechnung), dann in einer
Schleife alle Nachbarn des aktuellen Punktes durchgehen und den besten merken, der **echt
näher** liegt als die bisher beste Distanz. Gibt es keinen, ist die Suche fertig.

*Tipp: Zwei Variablen reichen — `beste` für die kleinste bisher gesehene Distanz und
`naechster` für den zugehörigen Index. Bleibt `naechster` nach einer Runde `None`, wird
abgebrochen. Jeder Aufruf von `distanz()` zählt, auch der auf Punkte, zu denen nicht gegangen
wird.*


In [ ]:
def suche_greedy(graph, vektoren, ziel, einstieg):
    """Geht vom Einstieg immer zum Nachbarn mit der kleinsten Distanz zum Ziel."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 2: suche_greedy() implementieren")


In [ ]:
# ✅ Selbsttest
gefunden, pfad, berechnungen = suche_greedy(KETTE, KETTE_PUNKTE, kreis(100)[0], einstieg=0)
assert gefunden == 5, f"Die Kette endet bei Punkt 5, nicht bei {gefunden}"
assert pfad == [0, 1, 2, 3, 4, 5], f"Jeder Punkt der Kette wird betreten: {pfad}"
assert berechnungen == 11, f"1 Einstieg + 10 Nachbarn = 11 Berechnungen, gezählt: {berechnungen}"

gefunden, pfad, berechnungen = suche_greedy(FALLE, FALLE_PUNKTE, ZIEL_FALLE, einstieg=0)
assert gefunden == 1, "In der Falle bleibt Greedy bei Punkt 1 stehen"
assert pfad == [0, 1], f"Zwei Punkte, dann ist Schluss: {pfad}"
assert berechnungen == 4, f"1 + 1 + 2 = 4 Berechnungen, gezählt: {berechnungen}"

gefunden, pfad, berechnungen = suche_greedy(GRAPH, VEKTOREN, VEKTOREN[42], einstieg=0)
assert gefunden == 42, "Auf den echten Daten findet Greedy hier den Punkt selbst"
assert berechnungen < len(VEKTOREN), "Weniger Berechnungen als die lineare Suche"

print("✅ Challenge 2 gelöst")
print(f"Anfrage {chunks[42]['chunk_id']}: {berechnungen} statt {len(VEKTOREN)} Distanzberechnungen")
print("Pfad:")
for i in pfad:
    print(f"  {i:>3}  {distanz(VEKTOREN[i], VEKTOREN[42]):.4f}  {chunks[i]['chunk_id']}")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def suche_greedy(graph, vektoren, ziel, einstieg):
    """Geht vom Einstieg immer zum Nachbarn mit der kleinsten Distanz zum Ziel."""
    aktuell = einstieg
    beste = distanz(vektoren[aktuell], ziel)
    berechnungen = 1
    pfad = [aktuell]

    while True:
        naechster = None
        for nachbar in graph[aktuell]:
            d = distanz(vektoren[nachbar], ziel)
            berechnungen += 1
            if d < beste:
                beste, naechster = d, nachbar

        if naechster is None:
            return aktuell, pfad, berechnungen

        aktuell = naechster
        pfad.append(aktuell)
```

`beste` wird innerhalb der Nachbarschleife fortgeschrieben. Dadurch bleibt am Ende der Runde
der beste Nachbar übrig und nicht der erste, der besser war als der Ausgangspunkt.

</details>


---
## 4 · Das lokale Minimum

📖 Greedy hält an, sobald kein Nachbar näher liegt. Das heißt nicht, dass es keinen näheren
Punkt gibt — es heißt nur, dass keiner der **Nachbarn** näher liegt. Genau das ist in der Falle
oben passiert: Der beste Punkt hing hinter einem schlechteren, und Greedy geht keinen Schritt
zurück.

▶️ Auf den 90 Chunks lässt sich das zählen. Jeder Punkt wird einmal als Anfrage benutzt; die
richtige Antwort ist immer der Punkt selbst, denn seine Distanz zu sich ist 0.


In [ ]:
# ▶️ Wo Greedy hängen bleibt
fehlschlaege = []
for i in range(len(VEKTOREN)):
    gefunden, pfad, berechnungen = suche_greedy(GRAPH, VEKTOREN, VEKTOREN[i], einstieg=0)
    if gefunden != i:
        fehlschlaege.append((i, gefunden, pfad, berechnungen))

print(f"{len(VEKTOREN) - len(fehlschlaege)} von {len(VEKTOREN)} Anfragen landen exakt richtig")
print()
for i, gefunden, pfad, berechnungen in fehlschlaege:
    print(f"gesucht:  {chunks[i]['chunk_id']}")
    print(f"gefunden: {chunks[gefunden]['chunk_id']}  "
          f"(Distanz {distanz(VEKTOREN[gefunden], VEKTOREN[i]):.4f} statt 0)")
    print(f"Pfad {pfad}, {berechnungen} Distanzberechnungen")
    print()


📖 Zwei von 90 Anfragen enden auf dem falschen Punkt. Beide Male ist der gefundene Punkt
inhaltlich nicht abwegig — `cve-2026-4410#00` statt `cve-2026-1187#00` sind zwei
CVE-Advisories — aber es ist die falsche Antwort, und niemand sagt es der Anwendung.

Die Abhilfe steckt schon im Wort *greedy*: Es wird immer nur **ein** Punkt weiterverfolgt. Statt
dessen hält die Suche eine **Kandidatenliste** der Größe `ef` (*exploration factor*):

* Die Liste enthält die `ef` bisher besten Punkte.
* Abgearbeitet wird immer der beste noch nicht geprüfte Kandidat — auch wenn er schlechter ist
  als der aktuell beste Treffer.
* Ein Nachbar kommt in die Liste, wenn sie noch nicht voll ist oder er besser ist als ihr
  schlechtester Eintrag.
* Schluss ist, wenn der beste offene Kandidat schlechter ist als der schlechteste Eintrag der
  vollen Liste. Dann kann über ihn nichts Besseres mehr kommen.

Mit `ef=1` ist das wieder Greedy. Je größer `ef`, desto mehr Umwege werden mitgenommen — und
desto mehr Distanzen werden gerechnet. `ef` ist der Regler zwischen Genauigkeit und Aufwand.


### 🛠️ Challenge 3: Suche mit Kandidatenliste

Schreibe `suche_beam(graph, vektoren, ziel, einstieg, ef=8)`. Rückgabe ist ein Tupel aus der
**Liste der besten Indizes** (aufsteigend nach Distanz, höchstens `ef` Stück) und der Zahl der
Distanzberechnungen.

Drei Datenstrukturen:

| Name | Inhalt |
|---|---|
| `offen` | Min-Heap der noch abzuarbeitenden Kandidaten als `(distanz, index)` |
| `beste` | sortierte Liste der bisher besten `ef` Punkte, ebenfalls als `(distanz, index)` |
| `gesehen` | Menge aller Punkte, deren Distanz schon berechnet wurde |

Der Ablauf:

1. Den Einstieg in alle drei Strukturen aufnehmen, `berechnungen = 1`.
2. Solange `offen` nicht leer ist: mit `heapq.heappop(offen)` den besten Kandidaten holen.
   Ist `beste` voll (`len(beste) >= ef`) und der Kandidat schlechter als `beste[-1]`, dann
   `break`.
3. Jeden noch nicht gesehenen Nachbarn messen und in `gesehen` eintragen.
4. Ist `beste` noch nicht voll oder der Nachbar besser als `beste[-1]`, kommt er mit
   `heapq.heappush` nach `offen` und in `beste`. Danach `beste.sort()` und `del beste[ef:]`.
5. Zurück kommen die Indizes aus `beste`, in dieser Reihenfolge.

*Tipp: Tupel `(distanz, index)` sortieren von selbst nach der Distanz — dafür braucht `heapq`
keinen Schlüssel.*


In [ ]:
def suche_beam(graph, vektoren, ziel, einstieg, ef=8):
    """Wie die Greedy-Suche, aber mit einer Kandidatenliste der Größe ef."""
    erste = distanz(vektoren[einstieg], ziel)
    berechnungen = 1

    offen = [(erste, einstieg)]      # noch abzuarbeiten, bester zuerst
    beste = [(erste, einstieg)]      # die bisher besten ef Punkte
    gesehen = {einstieg}

    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 3: suche_beam() implementieren")


In [ ]:
# ✅ Selbsttest
treffer, berechnungen = suche_beam(FALLE, FALLE_PUNKTE, ZIEL_FALLE, einstieg=0, ef=1)
assert treffer == [1], f"Mit ef=1 endet die Suche wie Greedy bei Punkt 1, nicht bei {treffer}"

treffer, berechnungen = suche_beam(FALLE, FALLE_PUNKTE, ZIEL_FALLE, einstieg=0, ef=3)
assert treffer[0] == 3, f"Mit ef=3 wird der Umweg über Punkt 2 mitgenommen, gefunden: {treffer}"
assert len(treffer) == 3, "Die Liste ist höchstens ef lang"
assert berechnungen == 4, f"Vier Punkte, vier Distanzen, gezählt: {berechnungen}"

treffer, berechnungen = suche_beam(GRAPH, VEKTOREN, VEKTOREN[5], einstieg=0, ef=10)
assert treffer[0] == 5, "Auch der Punkt, an dem Greedy scheitert, wird gefunden"
abstaende = [distanz(VEKTOREN[t], VEKTOREN[5]) for t in treffer]
assert abstaende == sorted(abstaende), "Die Trefferliste ist nach Distanz sortiert"

wahr = suche_linear(VEKTOREN, VEKTOREN[5], k=5)[0]
assert set(suche_beam(GRAPH, VEKTOREN, VEKTOREN[5], 0, ef=20)[0][:5]) == set(wahr), \
    "Mit ef=20 stehen dieselben fünf Nachbarn da wie bei der linearen Suche"

print("✅ Challenge 3 gelöst")
print(f"{chunks[5]['chunk_id']} — gefunden mit {berechnungen} statt {len(VEKTOREN)} Berechnungen")
for rang, i in enumerate(treffer[:5], start=1):
    print(f"  {rang}. {chunks[i]['chunk_id']:<32} Distanz {distanz(VEKTOREN[i], VEKTOREN[5]):.4f}")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def suche_beam(graph, vektoren, ziel, einstieg, ef=8):
    """Wie die Greedy-Suche, aber mit einer Kandidatenliste der Größe ef."""
    erste = distanz(vektoren[einstieg], ziel)
    berechnungen = 1

    offen = [(erste, einstieg)]      # noch abzuarbeiten, bester zuerst
    beste = [(erste, einstieg)]      # die bisher besten ef Punkte
    gesehen = {einstieg}

    while offen:
        d, knoten = heapq.heappop(offen)
        if len(beste) >= ef and d > beste[-1][0]:
            break

        for nachbar in graph[knoten]:
            if nachbar in gesehen:
                continue
            gesehen.add(nachbar)
            dn = distanz(vektoren[nachbar], ziel)
            berechnungen += 1

            if len(beste) < ef or dn < beste[-1][0]:
                heapq.heappush(offen, (dn, nachbar))
                beste.append((dn, nachbar))
                beste.sort()
                del beste[ef:]

    return [knoten for _, knoten in beste], berechnungen
```

`gesehen` ist nicht nur Buchhaltung: Ohne die Menge würde derselbe Punkt über verschiedene
Kanten mehrfach gemessen, und die Zahl der Distanzberechnungen liefe davon.

Die Abbruchbedingung ist die eigentliche Ersparnis. Ohne sie läuft die Suche durch den ganzen
Graphen, weil immer noch irgendein Kandidat offen ist.

</details>


---
## 5 · Die Hierarchie

📖 **Hierarchical NSW** legt mehrere Graphen übereinander:

* Die unterste Ebene enthält **alle** Punkte, fein vernetzt.
* Jede Ebene darüber enthält nur einen Bruchteil davon — üblich ist jeder achte. Dieselben
  `m` Kanten je Punkt spannen dort viel weitere Strecken, weil die Punkte dünner stehen.
* Ganz oben liegen ein paar Dutzend Punkte, die über die gesamte Sammlung verteilt sind.

Gesucht wird von oben nach unten. Auf jeder Ebene läuft dieselbe Suche wie eben, und der beste
gefundene Punkt ist der **Einstieg in die Ebene darunter**. Oben werden mit wenigen Schritten
große Strecken zurückgelegt, unten wird nur noch feinjustiert.

```
Ebene 1   ●───────────●───────────●        wenige Punkte, weite Sprünge
              │
Ebene 2   ●───●───●───●───●───●───●        Einstieg von oben
                  │
Ebene 3   ●─●─●─●─●─●─●─●─●─●─●─●─●        alle Punkte, feine Suche
```

Damit ersetzt die Hierarchie die zufälligen Fernverbindungen: Die oberen Ebenen **sind** die
Fernverbindungen, nur geordnet statt zufällig. Die Ebenen dieses Notebooks werden deshalb mit
`fern=0` gebaut.


### 🛠️ Challenge 4: Ebenen bauen und von oben nach unten suchen

Zwei Funktionen.

**`baue_hnsw(vektoren, m=6, layer=4, anteil=8, seed=0)`** gibt eine **Liste von Ebenen**
zurück. `ebenen[0]` ist die oberste und dünnste, `ebenen[-1]` enthält alle Punkte. Jede Ebene
ist ein Graph in derselben Form wie bisher — nur stehen in den Schlüsseln die Indizes aus
`vektoren`, nicht die Positionen innerhalb der Ebene.

1. Mit allen Punkten anfangen: `punkte = list(range(len(vektoren)))`.
2. Höchstens `layer` Ebenen bauen. Ab der zweiten wird ausgedünnt: Jeder Punkt bleibt mit
   Wahrscheinlichkeit `1 / anteil` drin (`rng.random() < 1 / anteil`). Bleiben `m` oder weniger
   Punkte übrig, wird abgebrochen — für einen Graphen sind das zu wenige.
3. Den Graphen einer Ebene baust du mit `baue_nsw(vektoren[punkte], m=m, fern=0, seed=seed + stufe)`.
   Er nummeriert von 0 durch; die Indizes müssen zurückübersetzt werden:
   `{punkte[i]: [punkte[j] for j in nachbarn] for i, nachbarn in teil.items()}`.
4. Gebaut wird von unten nach oben, zurückgegeben von oben nach unten — also am Ende
   `ebenen.reverse()`.

**`suche_hnsw(ebenen, vektoren, ziel, ef=8, einstieg=None)`** gibt die Trefferliste, die Liste
der Einstiege je Ebene und die Zahl der Distanzberechnungen zurück. Ohne Angabe ist der
Einstieg der erste Punkt der obersten Ebene (`next(iter(ebenen[0]))`).

Für jede Ebene von oben nach unten: `suche_beam` aufrufen, die Berechnungen aufaddieren, den
besten Treffer als Einstieg für die nächste Ebene merken und in der Liste der Einstiege
festhalten. In den oberen Ebenen genügt `ef=1` — dort wird nur navigiert. Nur die unterste
Ebene sucht mit dem übergebenen `ef`.


In [ ]:
def baue_hnsw(vektoren, m=6, layer=4, anteil=8, seed=0):
    """Baut bis zu `layer` Ebenen. Nach oben bleibt jeder `anteil`-te Punkt übrig."""
    rng = np.random.default_rng(seed)
    punkte = list(range(len(vektoren)))
    ebenen = []

    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 4: baue_hnsw() implementieren")


def suche_hnsw(ebenen, vektoren, ziel, ef=8, einstieg=None):
    """Sucht Ebene für Ebene von oben nach unten und reicht den besten Punkt weiter."""
    if einstieg is None:
        einstieg = next(iter(ebenen[0]))

    berechnungen = 0
    einstiege = []

    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 4: suche_hnsw() implementieren")


In [ ]:
# ✅ Selbsttest
EBENEN = baue_hnsw(VEKTOREN, m=6, layer=4, anteil=6, seed=0)
groessen = [len(e) for e in EBENEN]

assert len(EBENEN) >= 2, "Mindestens zwei Ebenen"
assert groessen == sorted(groessen), f"Nach oben wird es dünner, nicht dichter: {groessen}"
assert groessen[0] < groessen[-1], "Die oberste Ebene ist kleiner als die unterste"
assert len(EBENEN[-1]) == len(VEKTOREN), "Die unterste Ebene enthält alle Punkte"
for oben, unten in zip(EBENEN, EBENEN[1:]):
    assert set(oben) <= set(unten), "Jede Ebene ist eine Teilmenge der Ebene darunter"
assert all(nachbar in ebene for ebene in EBENEN for k in ebene for nachbar in ebene[k]), \
    "Die Nachbarn eines Punktes liegen in derselben Ebene"

treffer, einstiege, berechnungen = suche_hnsw(EBENEN, VEKTOREN, VEKTOREN[5], ef=10)
assert len(einstiege) == len(EBENEN), "Jede Ebene gibt einen Einstieg an die nächste weiter"
assert all(e in ebene for e, ebene in zip(einstiege, EBENEN)), "Jeder Einstieg liegt in seiner Ebene"
assert treffer[0] == 5, "Auch über die Ebenen wird der richtige Punkt gefunden"
assert berechnungen < len(VEKTOREN), "Weniger Berechnungen als die lineare Suche"

print("✅ Challenge 4 gelöst")
print(f"{len(EBENEN)} Ebenen mit {groessen} Punkten")
print()
print(f"Anfrage {chunks[5]['chunk_id']}, {berechnungen} Distanzberechnungen:")
for nummer, (ebene, punkt) in enumerate(zip(EBENEN, einstiege), start=1):
    print(f"  Ebene {nummer} ({len(ebene):>3} Punkte)  →  {chunks[punkt]['chunk_id']:<32}"
          f" Distanz {distanz(VEKTOREN[punkt], VEKTOREN[5]):.4f}")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_hnsw(vektoren, m=6, layer=4, anteil=8, seed=0):
    """Baut bis zu `layer` Ebenen. Nach oben bleibt jeder `anteil`-te Punkt übrig."""
    rng = np.random.default_rng(seed)
    punkte = list(range(len(vektoren)))
    ebenen = []

    for stufe in range(layer):
        if stufe > 0:
            punkte = [i for i in punkte if rng.random() < 1 / anteil]
            if len(punkte) <= m:
                break

        teil = baue_nsw(vektoren[punkte], m=m, fern=0, seed=seed + stufe)
        ebenen.append({punkte[i]: [punkte[j] for j in nachbarn]
                       for i, nachbarn in teil.items()})

    ebenen.reverse()
    return ebenen


def suche_hnsw(ebenen, vektoren, ziel, ef=8, einstieg=None):
    """Sucht Ebene für Ebene von oben nach unten und reicht den besten Punkt weiter."""
    if einstieg is None:
        einstieg = next(iter(ebenen[0]))

    berechnungen = 0
    einstiege = []
    treffer = [einstieg]

    for nummer, graph in enumerate(ebenen):
        unterste = nummer == len(ebenen) - 1
        treffer, kosten = suche_beam(graph, vektoren, ziel, einstieg,
                                     ef=ef if unterste else 1)
        berechnungen += kosten
        einstieg = treffer[0]
        einstiege.append(einstieg)

    return treffer, einstiege, berechnungen
```

Der Abbruch bei `len(punkte) <= m` ist nötig, weil `baue_nsw` sonst mehr Nachbarn sucht, als es
Punkte gibt. Bei 90 Vektoren und `anteil=6` reicht es deshalb nur für zwei Ebenen.

Die echten Implementierungen ziehen die Ebene eines Punktes beim Einfügen aus einer
exponentiellen Verteilung, statt in jeder Runde neu zu würfeln. Das Ergebnis ist dasselbe: Die
Zahl der Punkte fällt nach oben geometrisch.

</details>


---
## 6 · Messen statt behaupten

📖 Vier Verfahren, drei Kennzahlen. Gemessen wird über alle 90 Punkte als Anfrage; die Wahrheit
liefert die lineare Suche.

| Kennzahl | Bedeutung |
|---|---|
| **Recall@1** | Anteil der Anfragen, bei denen der erste Treffer der richtige ist |
| **Recall@5** | Anteil der fünf echten Nachbarn, die unter den ersten fünf Treffern stehen |
| **Berechnungen** | Aufrufe von `distanz()` je Anfrage |

Ein Hinweis zu Recall@5: Eine Kandidatenliste der Größe `ef` kann nicht mehr als `ef` Punkte
zurückgeben. Mit `ef < 5` ist Recall@5 deshalb rechnerisch gedeckelt. `ef` muss mindestens so
groß sein wie die Zahl der gesuchten Nachbarn — die Bibliotheken erzwingen das.


In [ ]:
# ▶️ Die Messung
def wahrheit_linear(vektoren, anfragen, k=5):
    """Die k echten Nachbarn jeder Anfrage, mit linearer Suche bestimmt."""
    return np.argsort(-(anfragen @ vektoren.T), axis=1)[:, :k]


def messe(anfragen, wahrheit, suche, k=5):
    """Vergleicht eine Suchfunktion mit der Wahrheit.

    `suche` bekommt einen Zielvektor und gibt (indizes, berechnungen) zurück.
    Rückgabe: Recall@1, Recall@k und Distanzberechnungen je Anfrage.
    """
    treffer_1 = treffer_k = berechnungen = 0
    for nummer, ziel in enumerate(anfragen):
        gefunden, kosten = suche(ziel)
        berechnungen += kosten
        soll = list(wahrheit[nummer])
        treffer_1 += int(bool(gefunden) and gefunden[0] == soll[0])
        treffer_k += len(set(gefunden[:k]) & set(soll)) / k
    n = len(anfragen)
    return treffer_1 / n, treffer_k / n, berechnungen / n


def greedy_als_liste(graph, vektoren, ziel):
    """Bringt das Ergebnis von suche_greedy() auf dieselbe Form wie die anderen Verfahren."""
    gefunden, _pfad, berechnungen = suche_greedy(graph, vektoren, ziel, 0)
    return [gefunden], berechnungen


def vergleiche(vektoren, anfragen, graph, ebenen, ef_werte):
    """Lineare Suche, Greedy, NSW und HNSW über dieselben Anfragen."""
    wahrheit = wahrheit_linear(vektoren, anfragen)
    ergebnis = {
        "linear": messe(anfragen, wahrheit, lambda z: suche_linear(vektoren, z, k=5)),
        "greedy": messe(anfragen, wahrheit, lambda z: greedy_als_liste(graph, vektoren, z)),
        "NSW": {}, "HNSW": {},
    }
    for ef in ef_werte:
        ergebnis["NSW"][ef] = messe(
            anfragen, wahrheit, lambda z, ef=ef: suche_beam(graph, vektoren, z, 0, ef))
        ergebnis["HNSW"][ef] = messe(
            anfragen, wahrheit, lambda z, ef=ef: suche_hnsw(ebenen, vektoren, z, ef)[0::2])
    return ergebnis


def zeige_tabelle(ergebnis, n):
    """Druckt das Ergebnis von vergleiche() als Tabelle."""
    kopf = f"{'Verfahren':<16}{'Recall@1':>10}{'Recall@5':>10}{'Berechnungen':>14}{'Anteil':>9}"
    print(kopf)
    print("-" * len(kopf))
    for name in ["linear", "greedy"]:
        r1, r5, kosten = ergebnis[name]
        print(f"{name:<16}{r1:>10.2f}{r5:>10.2f}{kosten:>14.0f}{kosten / n:>8.0%}")
    for verfahren in ["NSW", "HNSW"]:
        print("-" * len(kopf))
        for ef, (r1, r5, kosten) in ergebnis[verfahren].items():
            print(f"{verfahren + ' ef=' + str(ef):<16}{r1:>10.2f}{r5:>10.2f}"
                  f"{kosten:>14.0f}{kosten / n:>8.0%}")


print("wahrheit_linear(), messe(), vergleiche() und zeige_tabelle() stehen bereit")


In [ ]:
# ▶️ Alle drei Verfahren über die 90 Chunks
EF_WERTE = [1, 5, 10, 20, 40]

ergebnis_chunks = vergleiche(VEKTOREN, VEKTOREN, GRAPH, EBENEN, EF_WERTE)
zeige_tabelle(ergebnis_chunks, len(VEKTOREN))


In [ ]:
# ▶️ Dasselbe als Diagramm
def log_achse(achse, werte):
    """Logarithmische y-Achse mit lesbaren Beschriftungen für den gegebenen Wertebereich."""
    unten, oben = min(werte), max(werte)
    fein = [10, 20, 30, 50, 70, 100, 200, 300, 500, 700, 1_000, 2_000, 3_000, 5_000, 10_000]
    grob = [10, 30, 100, 300, 1_000, 3_000, 10_000]
    marken = [t for t in (grob if oben / unten > 20 else fein)
              if unten * 0.8 <= t <= oben * 1.25]
    achse.set_yscale("log")
    achse.set_yticks(marken, [f"{t:,}".replace(",", ".") for t in marken])
    achse.yaxis.set_minor_formatter(ticker.NullFormatter())


def zeige_kurven(ergebnis, n, titel):
    """Recall und Rechenaufwand über wachsendem ef."""
    ef = list(ergebnis["NSW"])
    bild, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4.2))

    for name, farbe in [("NSW", ORANGE), ("HNSW", BLAU)]:
        links.plot(ef, [ergebnis[name][e][0] for e in ef], "o-", color=farbe, label=name)
    links.axhline(1.0, color=GRAU, linestyle="--", label="lineare Suche")
    links.set_xscale("log", base=2)
    links.set_xticks(ef, [str(e) for e in ef])
    links.set_ylim(0, 1.05)
    links.set_xlabel("ef")
    links.set_ylabel("Recall@1")
    links.set_title("Genauigkeit")
    links.legend(loc="lower right")

    for name, farbe in [("NSW", ORANGE), ("HNSW", BLAU)]:
        rechts.plot(ef, [ergebnis[name][e][2] for e in ef], "o-", color=farbe, label=name)
    rechts.axhline(n, color=GRAU, linestyle="--", label="lineare Suche")
    rechts.set_xscale("log", base=2)
    rechts.set_xticks(ef, [str(e) for e in ef])
    log_achse(rechts, [ergebnis[name][e][2] for name in ("NSW", "HNSW") for e in ef] + [n])
    rechts.set_xlabel("ef")
    rechts.set_ylabel("Distanzberechnungen je Anfrage")
    rechts.set_title("Aufwand")
    rechts.legend(loc="upper left")

    bild.suptitle(titel, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


zeige_kurven(ergebnis_chunks, len(VEKTOREN), "90 Chunks: der Index gewinnt fast nichts")


📖 Der Befund ist unspektakulär, und das ist die Aussage: **Bei 90 Vektoren lohnt sich der Index
nicht.** Mit `ef=5` steht der Recall bei 1,0, und der Aufwand liegt bei rund 60 Prozent der
linearen Suche. Bei `ef=40` rechnet HNSW mehr Distanzen, als die lineare Suche überhaupt hat —
die Ebenen kosten Verwaltung, die sich hier nicht auszahlt.

Dazu kommt: Bei 90 Punkten und `anteil=6` reicht es nur für zwei Ebenen. Die Hierarchie kann
gar nicht zeigen, was sie kann.

Um den Effekt zu sehen, braucht es mehr Punkte.


---
## 7 · Derselbe Test mit 8.000 Vektoren

📖 Echte Embeddings sind nicht gleichmäßig im Raum verteilt: Texte zum selben Thema liegen
beieinander, dazwischen ist es leer. Der synthetische Datensatz bildet das nach — 30 zufällige
Zentren, um die herum gestreut wird. Alles hängt an einem festen `seed`, das Ergebnis ist bei
jedem Durchlauf dasselbe.

24 Dimensionen statt 768, damit die Rechnung im Notebook in Sekunden läuft. An der Sache ändert
das nichts: Gezählt werden Distanzberechnungen, nicht Multiplikationen.

▶️ Datensatz erzeugen, Graph und Ebenen bauen.


In [ ]:
# ▶️ 8.000 Vektoren um 30 Zentren, dazu 200 Anfragen aus derselben Verteilung
def erzeuge_daten(n, anfragen=200, dim=24, zentren=30, streuung=0.4, seed=7):
    """Zufällige Vektoren um `zentren` Häufungspunkte, alle auf Länge 1."""
    rng = np.random.default_rng(seed)
    mitte = rng.normal(size=(zentren, dim))
    mitte /= np.linalg.norm(mitte, axis=1, keepdims=True)

    alle = (mitte[rng.integers(0, zentren, size=n + anfragen)]
            + streuung * rng.normal(size=(n + anfragen, dim)))
    alle = (alle / np.linalg.norm(alle, axis=1, keepdims=True)).astype(np.float32)
    return alle[:n], alle[n:]


PUNKTE, ANFRAGEN = erzeuge_daten(8_000)

t0 = time.perf_counter()
GRAPH_GROSS = baue_nsw(PUNKTE, m=8, fern=2, seed=0)
EBENEN_GROSS = baue_hnsw(PUNKTE, m=8, layer=5, anteil=8, seed=0)
bauzeit = time.perf_counter() - t0

kanten = sum(len(v) for v in GRAPH_GROSS.values()) // 2
print(f"{len(PUNKTE):,}".replace(",", ".")
      + f" Vektoren à {PUNKTE.shape[1]} Dimensionen, {len(ANFRAGEN)} Anfragen")
print(f"NSW: {kanten:,}".replace(",", ".")
      + f" Kanten, Median {int(np.median([len(v) for v in GRAPH_GROSS.values()]))} Nachbarn je Punkt")
print(f"HNSW: {len(EBENEN_GROSS)} Ebenen mit {[len(e) for e in EBENEN_GROSS]} Punkten")
print(f"Bauzeit für beide Indizes: {bauzeit:.1f} s")


In [ ]:
# ▶️ Dieselbe Messung, dieselbe Tabelle
EF_GROSS = [1, 5, 10, 20, 40, 80]

ergebnis_gross = vergleiche(PUNKTE, ANFRAGEN, GRAPH_GROSS, EBENEN_GROSS, EF_GROSS)
zeige_tabelle(ergebnis_gross, len(PUNKTE))


In [ ]:
# ▶️ Und dasselbe Diagramm
zeige_kurven(ergebnis_gross, len(PUNKTE), "8.000 Vektoren: der Index rechnet ein Zwanzigstel")


📖 Jetzt ist der Unterschied da. Mit `ef=40` findet HNSW bei rund 90 Prozent der Anfragen genau
den richtigen ersten Nachbarn und braucht dafür etwa 390 statt 8.000 Distanzberechnungen — ein
Zwanzigstel. Wer 95 Prozent will, stellt `ef=80` und zahlt rund 650.

Auch der Unterschied zwischen NSW und HNSW ist sichtbar: Bei gleichem `ef` und praktisch
gleichem Recall rechnet HNSW rund 30 Prozent weniger. Die geordneten Ebenen bringen die Suche
schneller in die richtige Gegend als die zufälligen Fernverbindungen.

Und die Greedy-Zeile ganz oben zeigt, was aus dem lokalen Minimum wird, wenn die Sammlung
wächst: Bei 90 Vektoren fand Greedy 98 Prozent der Anfragen, bei 8.000 sind es 13 Prozent. Ohne
Kandidatenliste ist eine Graph-Suche in einer großen Sammlung unbrauchbar.

▶️ Die letzte Messung: Wie wächst der Aufwand, wenn die Sammlung wächst?


In [ ]:
# ▶️ Aufwand über wachsender Sammlung, ef fest auf 40
GROESSEN = [500, 1_000, 2_000, 4_000, 8_000]
EF_FEST = 40

verlauf = {"NSW": [], "HNSW": []}
recall = {"NSW": [], "HNSW": []}

for groesse in GROESSEN:
    teil = PUNKTE[:groesse]
    wahrheit = wahrheit_linear(teil, ANFRAGEN)
    graph = baue_nsw(teil, m=8, fern=2, seed=0)
    ebenen = baue_hnsw(teil, m=8, layer=5, anteil=8, seed=0)

    for name, suche in [("NSW", lambda z: suche_beam(graph, teil, z, 0, EF_FEST)),
                        ("HNSW", lambda z: suche_hnsw(ebenen, teil, z, EF_FEST)[0::2])]:
        r1, r5, kosten = messe(ANFRAGEN, wahrheit, suche)
        verlauf[name].append(kosten)
        recall[name].append(r1)

print(f"{'Sammlung':>10}{'linear':>10}{'NSW':>10}{'HNSW':>10}"
      f"{'Recall@1 NSW':>15}{'Recall@1 HNSW':>15}")
print("-" * 70)
for i, groesse in enumerate(GROESSEN):
    print(f"{groesse:>10,}".replace(",", ".")
          + f"{groesse:>10,}".replace(",", ".")
          + f"{verlauf['NSW'][i]:>10.0f}{verlauf['HNSW'][i]:>10.0f}"
          + f"{recall['NSW'][i]:>15.2f}{recall['HNSW'][i]:>15.2f}")


In [ ]:
# ▶️ Die Kurve dazu
plt.figure(figsize=(8, 4.5))
plt.loglog(GROESSEN, GROESSEN, "o--", color=GRAU, label="lineare Suche")
plt.loglog(GROESSEN, verlauf["NSW"], "o-", color=ORANGE, label=f"NSW, ef={EF_FEST}")
plt.loglog(GROESSEN, verlauf["HNSW"], "o-", color=BLAU, label=f"HNSW, ef={EF_FEST}")
achse = plt.gca()
achse.set_xticks(GROESSEN, [f"{g:,}".replace(",", ".") for g in GROESSEN])
achse.xaxis.set_minor_formatter(ticker.NullFormatter())
log_achse(achse, verlauf["HNSW"] + [max(GROESSEN)])
plt.xlabel("Vektoren in der Sammlung")
plt.ylabel("Distanzberechnungen je Anfrage")
plt.title("Sechzehnfache Sammlung, nicht einmal doppelter Aufwand")
plt.legend()
plt.show()


📖 Die graue Gerade ist die lineare Suche: sechzehnmal so viele Vektoren, sechzehnmal so viele
Berechnungen. HNSW steigt im selben Bereich von 239 auf 392, NSW von 305 auf 547 — Faktor 1,6
und 1,8 statt 16. Der Aufwand einer Graph-Suche hängt fast nicht an der Sammlungsgröße, sondern
an `ef` und an der Zahl der Nachbarn `m`.

Bezahlt wird das an zwei Stellen. Der Recall sinkt langsam, von 0,99 bei 500 Vektoren auf 0,91
bei 8.000 — bei festem `ef` wird die Suche mit wachsender Sammlung ungenauer. Und der Index
muss gebaut und im Speicher gehalten werden.

**Der Satz zum Mitnehmen: Ein Index tauscht Genauigkeit gegen Rechenaufwand, und `ef` ist der
Regler dafür.** Es gibt keine Einstellung, die beides gibt. Es gibt nur die Entscheidung, wie
viele Prozent Recall ein Prozent Latency wert sind.


---
## 8 · Dieselben Parameter in Chroma

📖 Chroma benutzt HNSW und legt genau die Regler frei, die in diesem Notebook gebaut wurden:

| Chroma | hier | Bedeutung |
|---|---|---|
| `hnsw:space` | `distanz()` | Abstandsmaß: `cosine`, `l2` oder `ip` |
| `hnsw:M` | `m` | Nachbarn je Punkt und Ebene. Mehr Kanten heißt besserer Recall, mehr Speicher, längerer Bau |
| `hnsw:construction_ef` | `ef` **beim Bauen** | wie gründlich beim Einfügen nach Nachbarn gesucht wird. Wirkt einmal, auf die Qualität des Graphen |
| `hnsw:search_ef` | `ef` **beim Suchen** | die Kandidatenliste einer Anfrage. Wirkt bei jeder Anfrage, auf Recall und Latency |

▶️ Vier Collections mit denselben 8.000 Vektoren, vier Parametersätze. Gemessen wird gegen
dieselbe Wahrheit wie oben.


In [ ]:
# ▶️ Chroma mit vier Parametersätzen
klient = chromadb.EphemeralClient()      # nur im Speicher, schreibt nichts auf die Platte
wahrheit_gross = wahrheit_linear(PUNKTE, ANFRAGEN)
kennungen = [str(i) for i in range(len(PUNKTE))]

zeilen = []

for m, bau_ef, such_ef in [(8, 20, 10), (8, 20, 100), (16, 100, 10), (16, 100, 100)]:
    t0 = time.perf_counter()
    sammlung = klient.create_collection(
        f"test-{m}-{bau_ef}-{such_ef}",
        metadata={"hnsw:space": "cosine", "hnsw:M": m,
                  "hnsw:construction_ef": bau_ef, "hnsw:search_ef": such_ef},
    )
    for start in range(0, len(PUNKTE), 2_000):
        block = PUNKTE[start:start + 2_000]
        sammlung.add(ids=kennungen[start:start + len(block)], embeddings=block.tolist())
    bauzeit = time.perf_counter() - t0

    t0 = time.perf_counter()
    antwort = sammlung.query(query_embeddings=ANFRAGEN.tolist(), n_results=5)
    anfragezeit = (time.perf_counter() - t0) / len(ANFRAGEN)

    gefunden = [[int(x) for x in zeile] for zeile in antwort["ids"]]
    r1 = np.mean([g[0] == w[0] for g, w in zip(gefunden, wahrheit_gross)])
    r5 = np.mean([len(set(g) & set(w)) / 5 for g, w in zip(gefunden, wahrheit_gross)])

    zeilen.append(f"{m:>8}{bau_ef:>18}{such_ef:>11}{bauzeit:>7.1f}s"
                  f"{anfragezeit * 1000:>8.2f}ms{r1:>10.3f}{r5:>10.3f}")

kopf = (f"{'hnsw:M':>8}{'construction_ef':>18}{'search_ef':>11}"
        f"{'Bau':>8}{'Anfrage':>10}{'Recall@1':>10}{'Recall@5':>10}")
print(kopf)
print("-" * len(kopf))
for zeile in zeilen:
    print(zeile)


📖 Die Tabelle zeigt beide Regler getrennt. Chroma zieht die Ebene eines Punktes ohne festen
`seed`, deshalb wackeln die Nachkommastellen zwischen zwei Durchläufen — die Größenordnungen
bleiben.

`hnsw:search_ef` von 10 auf 100 hebt den Recall@1 von rund zwei Dritteln auf über 0,97 und
kostet dafür Zeit je Anfrage. Das ist derselbe Regler wie `ef` oben — er lässt sich jederzeit
ändern, ohne den Index neu zu bauen.

`hnsw:M` und `hnsw:construction_ef` wirken auf den Graphen selbst. Der bessere Graph
(`M=16, construction_ef=100`) kommt schon mit `search_ef=10` deutlich weiter als der dünne, und
mit `search_ef=100` liefert er alle Nachbarn. Diese beiden Parameter stehen beim Anlegen der
Collection fest; sie zu ändern heißt, den Index neu zu bauen.

Wann fasst man sie an?

* **Recall zu niedrig, Latency egal** — `hnsw:search_ef` hochdrehen. Erste Maßnahme, kostet
  nichts außer Rechenzeit je Anfrage.
* **Recall auch dann zu niedrig** — `hnsw:M` und `hnsw:construction_ef` erhöhen und neu
  aufbauen. Kostet Speicher und Bauzeit.
* **Sammlung im fünfstelligen Bereich oder kleiner** — meistens gar nichts. Die Voreinstellungen
  von Chroma reichen, und die lineare Suche wäre auch schnell genug.


---
## 9 · Was du gebaut hast

* `baue_nsw()` — der Graph aus Nah- und Fernverbindungen. Die zufälligen Fernkanten senken die
  mittlere Weglänge, ohne die Nachbarschaften zu zerstören.
* `suche_greedy()` — die Suche, die immer zum besten sichtbaren Nachbarn geht. Zwei von 90
  Anfragen enden damit auf dem falschen Punkt: ein lokales Minimum.
* `suche_beam()` — dieselbe Suche mit einer Kandidatenliste der Größe `ef`. Damit werden auch
  Umwege mitgenommen, und der Recall lässt sich einstellen.
* `baue_hnsw()` und `suche_hnsw()` — mehrere Ebenen, oben dünn und weitmaschig, unten
  vollständig. Der beste Punkt einer Ebene ist der Einstieg in die nächste.
* `messe()` und `vergleiche()` — Recall@1, Recall@5 und Distanzberechnungen je Anfrage, über
  90 echte und 8.000 synthetische Vektoren.

Die Zahlen zum Mitnehmen. Bei **90 Vektoren** bringt der Index nichts: Recall 1,0 gibt es ab
`ef=5` für rund 60 Prozent des Aufwands, und bei `ef=40` rechnet HNSW mehr als die lineare Suche.
Bei **8.000 Vektoren** liefert HNSW mit `ef=40` einen Recall@1 von 0,91 für rund 390 statt
8.000 Distanzberechnungen. Und bei sechzehnfacher Sammlung steigt der Aufwand nur um den
Faktor 1,6, während der Recall langsam sinkt.

Das ist der Handel, den jede Vector Database anbietet: **approximativ statt exakt**. Ein
HNSW-Index findet nicht immer den nächsten Nachbarn — er findet fast immer einen der nächsten,
und zwar sehr schnell. Für Retrieval ist das der richtige Handel, weil hinter der Suche ohnehin
ein Sprachmodell steht und nicht eine Kontoführung.


---
### 🔬 Bonus — ohne Lösung

**1. Punkte löschen.** Eine Knowledge Base ändert sich: Dokumente werden zurückgezogen, Chunks
neu geschnitten. Aus einem Graphen einen Knoten zu entfernen ist heikler als ihn einzufügen.
Vorgehen:

1. Schreibe `loesche(graph, punkt)`. Der Knoten verschwindet, und er wird aus allen
   Nachbarlisten gestrichen.
2. Prüfe mit `erreichbar()`, ob der Graph danach noch zusammenhängend ist. Lösche gezielt die
   Punkte mit dem größten Grad und zähle, nach wie vielen Löschungen er zerfällt.
3. Baue die Reparatur ein: Die Nachbarn des gelöschten Punktes werden untereinander verbunden.
   Wie viele Kanten kostet das, und wie verändert sich der Recall?

Vergleiche das mit dem Weg, den echte Systeme gehen: Der Punkt bleibt im Graphen stehen und
wird nur als gelöscht markiert (*tombstone*), gefiltert wird erst im Ergebnis. Ab wann lohnt
sich ein vollständiger Neuaufbau?

**2. Produktquantisierung.** Der zweite Hebel für große Sammlungen ist nicht die Zahl der
Distanzberechnungen, sondern ihr Preis. Ein Vektor mit 768 float32-Werten belegt 3 Kilobyte;
eine Million Vektoren sind drei Gigabyte, die bei jeder Anfrage durch den Speicher wandern.
Produktquantisierung zerlegt jeden Vektor in Teilstücke und ersetzt jedes Teilstück durch die
Nummer eines Zentroids. Vorgehen:

1. Die 24 Dimensionen von `PUNKTE` in acht Blöcke zu je drei Dimensionen zerlegen.
2. Je Block 256 Zentroiden bestimmen — mit `sklearn.cluster.KMeans` oder einer eigenen
   k-Means-Schleife.
3. Jeden Vektor als acht Bytes speichern: je Block die Nummer des nächsten Zentroids. Aus 96
   Byte werden 8.
4. Die Distanz zwischen einer Anfrage und einem Code über eine vorberechnete Tabelle bestimmen:
   je Block die Distanz der Anfrage zu allen 256 Zentroiden, dann acht Nachschlagevorgänge
   addieren.

Miss Recall@1 gegen die exakte Distanz. Interessant ist die Kombination: Wie viel Recall bleibt
übrig, wenn HNSW auf quantisierten Vektoren sucht — und wie viel davon holt ein Re-Ranking der
Top-50 mit den echten Vektoren zurück?
